In [16]:
%pip install tqdm


Note: you may need to restart the kernel to use updated packages.


In [27]:
# Weaviate Data Loader Notebook con control de reproceso, logging y uso de variables de entorno

import weaviate
from weaviate.connect import ConnectionParams
from weaviate.auth import AuthApiKey
from weaviate.classes.config import Property, Configure, DataType
import json
import os
from datetime import datetime
from dotenv import load_dotenv
from tqdm import tqdm  # barra de progreso opcional

# === CARGA VARIABLES DE ENTORNO ===
load_dotenv()

WCS_URL = os.getenv("WCS_URL")
WCS_API_KEY = os.getenv("WCS_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

LOG_FILE = "weaviate_errors.log"
QA_PROCESSED_FILE = "qa_uploaded_ids.txt"
DOCS_PROCESSED_FILE = "docs_uploaded_links.txt"

# Validar que las variables estén cargadas
assert WCS_URL and WCS_API_KEY and OPENAI_API_KEY, "Faltan variables de entorno necesarias."

# Conexión al cluster Weaviate Cloud Service con nueva sintaxis
client = weaviate.connect_to_wcs(
    cluster_url=WCS_URL,
    auth_credentials=AuthApiKey(WCS_API_KEY),
    headers={"X-OpenAI-Api-Key": OPENAI_API_KEY}
)

# Verificamos la conexión
assert client.is_ready(), "Error: no se pudo conectar a Weaviate."

# === DEFINICIÓN DE ESQUEMAS ===
# Crear solo si no existen
class_names = client.collections.list_all()

if "QnA" not in class_names:
    client.collections.create(
        name="QnA",
        vectorizer_config=Configure.Vectorizer.text2vec_openai(),
        properties=[
            Property(name="title", data_type=DataType.TEXT),
            Property(name="question_content", data_type=DataType.TEXT),
            Property(name="accepted_answer", data_type=DataType.TEXT),
            Property(name="url", data_type=DataType.TEXT),
            Property(name="tags", data_type=DataType.TEXT_ARRAY),
            Property(name="date", data_type=DataType.DATE)
        ]
    )

if "Documentation" not in class_names:
    client.collections.create(
        name="Documentation",
        vectorizer_config=Configure.Vectorizer.text2vec_openai(),
        properties=[
            Property(name="title", data_type=DataType.TEXT),
            Property(name="summary", data_type=DataType.TEXT),
            Property(name="content", data_type=DataType.TEXT),
            Property(name="link", data_type=DataType.TEXT),
            Property(name="related_links", data_type=DataType.TEXT_ARRAY)
        ]
    )

# === FUNCIONES DE UTILIDAD ===
def log_error(msg):
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        timestamp = datetime.now().isoformat()
        f.write(f"[{timestamp}] {msg}\n")

# === CARGA DE PREGUNTAS ===
print("\nIniciando carga de preguntas...")
uploaded_qa_ids = set()
if os.path.exists(QA_PROCESSED_FILE):
    with open(QA_PROCESSED_FILE, "r", encoding="utf-8") as f:
        uploaded_qa_ids = set(line.strip() for line in f if line.strip())

with open("questions_data.json", "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(QA_PROCESSED_FILE, "a", encoding="utf-8") as log_qa:
    for i, line in enumerate(tqdm(lines, desc="Subiendo preguntas"), start=1):
        try:
            qa = json.loads(line)
            if not qa.get("title") or not qa.get("question_content") or not qa.get("url"):
                log_error(f"QnA Error [línea {i} con campos requeridos vacíos]")
                continue

            unique_id = qa["url"]
            if unique_id in uploaded_qa_ids:
                continue

            client.collections.get("QnA").data.insert(
                {
                    "title": qa["title"],
                    "question_content": qa["question_content"],
                    "accepted_answer": qa.get("accepted_answer", ""),
                    "url": qa["url"],
                    "tags": qa.get("tags", []),
                    "date": qa["date"]
                }
            )
            log_qa.write(f"{unique_id}\n")
            uploaded_qa_ids.add(unique_id)
        except json.JSONDecodeError as e:
            log_error(f"QnA Error [línea {i} malformada]: {e}")
        except Exception as e:
            url_info = qa["url"] if 'qa' in locals() and "url" in qa else "sin_url"
            log_error(f"QnA Error [{url_info}]: {e}")

# === CARGA DE DOCUMENTACIÓN ===
print("\nIniciando carga de documentación...")
uploaded_docs_links = set()
if os.path.exists(DOCS_PROCESSED_FILE):
    with open(DOCS_PROCESSED_FILE, "r", encoding="utf-8") as f:
        uploaded_docs_links = set(line.strip() for line in f if line.strip())

with open("azure_docs_full.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

with open(DOCS_PROCESSED_FILE, "a", encoding="utf-8") as log_docs:
    for i, doc in enumerate(tqdm(docs, desc="Subiendo docs"), start=1):
        try:
            if not doc.get("title") or not doc.get("content") or not doc.get("link"):
                log_error(f"DOC Error [línea {i} con campos requeridos vacíos]")
                continue

            unique_link = doc["link"]
            if unique_link in uploaded_docs_links:
                continue

            client.collections.get("Documentation").data.insert(
                {
                    "title": doc["title"],
                    "summary": doc.get("summary", ""),
                    "content": doc["content"],
                    "link": doc["link"],
                    "related_links": doc.get("related_links", [])
                }
            )
            log_docs.write(f"{unique_link}\n")
            uploaded_docs_links.add(unique_link)
        except Exception as e:
            log_error(f"DOC Error [línea {i}]: {e}")


Iniciando carga de preguntas...


Subiendo preguntas: 100%|██████████| 21767/21767 [11:54:15<00:00,  1.97s/it]      



Iniciando carga de documentación...


Subiendo docs:   2%|▏         | 716/41691 [01:21<05:52, 116.28it/s] /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/selector_events.py:879: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=88 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/asyncio/selector_events.py:879: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=85 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
Subiendo docs: 100%|██████████| 41691/41691 [1:08:06<00:00, 10.20it/s] 
